<a href="https://colab.research.google.com/github/Loperamids/maternal-risk-app/blob/main/Final_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 🔹 CELL 1 (RUN THIS FIRST)
!pip install streamlit pyngrok scikit-learn seaborn matplotlib pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 86.2 MB/s eta 0:00:00


In [2]:
!pip install streamlit
!pip install pyngrok

In [12]:
    import streamlit as st
    import pandas as pd
    import numpy as np
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        confusion_matrix,
        classification_report
    )
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    import joblib
    from datetime import datetime

    df1 = pd.read_excel("maternal_cleaned_bp-1.xlsx", engine='openpyxl')
    df2 = pd.read_excel("final_processed_dataset.xlsx", engine='openpyxl')

    # ✅ COMBINE FIRST
    df = pd.concat([df1, df2], ignore_index=True)

    df.columns = df.columns.str.strip().str.lower()

    # ✅ FIX COLUMN NAMES
    df = df.rename(columns={
        "body_temperature": "temperature",
        "systolic": "systolic_bp",
        "diastolic": "diastolic_bp",
        "pre-pregnancy_weight": "pre_pregnancy_weight",
        "risk_level": "risk"
    })

    # ✅ CONVERT TARGET (Low/High → 0/1)
    df["risk"] = df["risk"].map({
        "Low": 0,
        "High": 1,
        "Low Risk": 0,
        "High Risk": 1
    })

    # ✅ REMOVE UNUSED COLUMNS
    df = df.drop(columns=["id", "name"], errors="ignore")

    # ✅ FORCE CORRECT COLUMN ORDER
    df = df[[
        "age",
        "systolic_bp",
        "diastolic_bp",
        "blood_sugar",
        "temperature",
        "heart_rate",
        "maternal_weight",
        "pre_pregnancy_weight",
        "fetal_age",
        "risk"
    ]]

    df.dropna(subset=["risk"], inplace=True)

    print("Data preprocessing complete ✓")
    print(f"Dataset shape: {df.shape}")
    print(f"Risk distribution:\n{df['risk'].value_counts()}")

    # Train model and save
    X = df.drop("risk", axis=1)
    y = df["risk"]

    Xtr, Xte, ytr, yte = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    scaler = StandardScaler()
    Xtr_scaled = scaler.fit_transform(Xtr)
    Xte_scaled = scaler.transform(Xte)

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=5,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42
    )
    model.fit(Xtr_scaled, ytr)

    ypred = model.predict(Xte_scaled)

    metrics = {
        "accuracy": accuracy_score(yte, ypred),
        "precision": precision_score(yte, ypred),
        "recall": recall_score(yte, ypred),
        "f1": f1_score(yte, ypred)
    }

    # ✅ SAVE EVERYTHING
    joblib.dump(model, "maternal_model.pkl")
    joblib.dump(scaler, "scaler.pkl")
    joblib.dump(metrics, "metrics.pkl")
    joblib.dump(yte, "ytest.pkl")
    joblib.dump(ypred, "ypred.pkl")

    print("Model trained and saved ✓")
    print(f"Test metrics: {metrics}")

    # ========================= STREAMLIT APP =========================
    app_code = '''
    import streamlit as st
    import pandas as pd
    import numpy as np
    import os
    import joblib

    from datetime import datetime

    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        confusion_matrix,
        classification_report
    )

    import matplotlib.pyplot as plt
    import seaborn as sns

    st.set_page_config(
        page_title="Maternal Risk Rapid Assessment System",
        page_icon=":hospital:",
        layout="wide",
        initial_sidebar_state="expanded"
    )

    # ========================= INITIALIZE SESSION STATE =========================
    if "patient_verified" not in st.session_state:
        st.session_state.patient_verified = False
    if "verified_patient_name" not in st.session_state:
        st.session_state.verified_patient_name = ""
    if "verified_patient_id" not in st.session_state:
        st.session_state.verified_patient_id = ""

    # ========================= CSS =========================
    st.markdown("""
        <style>
        .main-header {font-size:2.5em;font-weight:bold;color:#2E8B57;text-align:center;}
        .sub-header {font-size:1.5em;font-weight:bold;color:#4682B4;margin-top:20px;}
        .risk-high {color:red;font-weight:bold;}
        .risk-low {color:green;font-weight:bold;}
        .info-box {background:#e7f3fe;border-left:6px solid #2196F3;padding:10px;}
        </style>
    """, unsafe_allow_html=True)


    # ========================= PATIENT ID =========================
    def generate_patient_id():
        file = "patient_records.csv"
        if os.path.exists(file):
            df = pd.read_csv(file)
            if not df.empty:
                nums = df["Patient_ID"].str.extract(r'P(\\d+)').dropna().astype(int)
                return f"P{nums.max().values[0]+1:04d}"
        return "P0001"
    # ========================= MODEL =========================

    @st.cache_resource
    def load_model():
        if os.path.exists("maternal_model.pkl") and os.path.exists("scaler.pkl"):
            model = joblib.load("maternal_model.pkl")
            scaler = joblib.load("scaler.pkl")

            if os.path.exists("metrics.pkl"):
                metrics = joblib.load("metrics.pkl")
                yte = joblib.load("ytest.pkl")
                ypred = joblib.load("ypred.pkl")
            else:
                metrics = {"accuracy": 0, "precision": 0, "recall": 0, "f1": 0}
                yte, ypred = [], []

            return model, scaler, metrics, yte, ypred

        else:
            st.error("Model files not found. Please run the training script first.")
            st.stop()
    # ===== LOAD MODEL =====
    model, scaler, metrics, yte, ypred = load_model()

    # ========================= TABS =========================
    tab1, tab2 = st.tabs(["Risk Assessment", "My Records"])

    # ========================= TAB 1 =========================
    with tab1:

        # ===== PATIENT VERIFICATION =====
        if not st.session_state.patient_verified:
            st.subheader("Patient Verification")

            ptype = st.radio("Patient Type", ["New Patient", "Existing Patient"])

            if ptype == "New Patient":
                pname = st.text_input("Patient Name", key="new_patient_name")

                if st.button("Register New Patient"):
                    if pname:
                        st.session_state.verified_patient_name = pname
                        st.session_state.verified_patient_id = generate_patient_id()
                        st.session_state.patient_verified = True
                        st.success(f"Assigned ID: {st.session_state.verified_patient_id}")
                        st.rerun()
                    else:
                        st.error("Enter patient name")

            else:
                pname = st.text_input("Patient Name", key="verify_name")
                pid = st.text_input("Patient ID", key="verify_id")

                if st.button("Proceed to Assessment"):
                    if os.path.exists("patient_records.csv"):
                        df = pd.read_csv("patient_records.csv")

                        match = df[
                            (df["Patient_ID"] == pid) &
                            (df["Patient_Name"].str.lower() == pname.lower())
                        ]

                        if not match.empty:
                            st.session_state.verified_patient_name = pname
                            st.session_state.verified_patient_id = pid
                            st.session_state.patient_verified = True
                            st.success("Patient verified")
                            st.rerun()
                        else:
                            st.error("Patient not found")
                    else:
                        st.error("No patient records file found")

            st.stop()

        # ===== CONTINUE ONLY IF VERIFIED =====
        st.subheader("Input Patient Data")

        patient_id = st.session_state.verified_patient_id
        patient_name = st.session_state.verified_patient_name

        st.text_input("Patient ID", patient_id, disabled=True)
        st.text_input("Patient Name", patient_name, disabled=True)

        col1, col2, col3 = st.columns(3)

        with col1:
            age = st.number_input("Age", value=25)  # removed min/max
            sbp = st.number_input("Systolic BP", value=115)
            dbp = st.number_input("Diastolic BP", value=75)

        with col2:
            sugar = st.number_input("Blood Sugar", value=95.0)
            temp = st.number_input("Temperature", value=37.0)
            hr = st.number_input("Heart Rate", value=78)

        with col3:
            mw = st.number_input("Maternal Weight", value=65.0)
            pw = st.number_input("Pre-Pregnancy Weight", value=60.0)
            fa = st.number_input("Fetal Age", value=28)

        # ===== PREDICTION =====
        if st.button("Assess Risk"):

            input_df = pd.DataFrame([{
                "age": age,
                "systolic_bp": sbp,
                "diastolic_bp": dbp,
                "blood_sugar": sugar,
                "temperature": temp,
                "heart_rate": hr,
                "maternal_weight": mw,
                "pre_pregnancy_weight": pw,
                "fetal_age": fa
            }])

            try:
                feature_order = scaler.feature_names_in_
                input_df = input_df[feature_order]
            except:
                pass

            input_scaled = scaler.transform(input_df)

            pred = model.predict(input_scaled)[0]

            risk = "High Risk" if pred == 1 else "Low Risk"

            if risk == "High Risk":
                st.error(f"Predicted Risk: {risk}")
            else:
                st.success(f"Predicted Risk: {risk}")

            st.markdown("---")
            st.subheader("Model Performance")

            st.markdown(f"""
                <div style="font-size:14px; line-height:1.6;">
                <b>Accuracy:</b> {metrics['accuracy']:.3f}<br>
                <b>Precision:</b> {metrics['precision']:.3f}<br>
                <b>Recall:</b> {metrics['recall']:.3f}<br>
                <b>F1-score:</b> {metrics['f1']:.3f}
                </div>
            """, unsafe_allow_html=True)

            record = pd.DataFrame([{
                "Patient_ID": patient_id,
                "Patient_Name": patient_name,
                "Age": age,
                "Systolic_BP": sbp,
                "Diastolic_BP": dbp,
                "Blood_Sugar": sugar,
                "Temperature": temp,
                "Heart_Rate": hr,
                "Maternal_Weight": mw,
                "Pre_Pregnancy_Weight": pw,
                "Fetal_Age": fa,
                "Risk": risk,
                "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }])

            if os.path.exists("patient_records.csv"):
                record = pd.concat([pd.read_csv("patient_records.csv"), record], ignore_index=True)

            record.to_csv("patient_records.csv", index=False)

        if st.button("Finish & New Patient"):
            st.session_state.patient_verified = False
            st.session_state.verified_patient_id = ""
            st.session_state.verified_patient_name = ""
            st.rerun()
    # ========================= TAB 2 =========================
    with tab2:
        st.subheader("My Records")
        ppid = st.text_input("Patient ID", key="search_pid")
        pname = st.text_input("Patient Name", key="search_name")
        if st.button("Search My Records"):
            if os.path.exists("patient_records.csv"):
                df = pd.read_csv("patient_records.csv")
                st.dataframe(df[
                    (df["Patient_ID"] == ppid) &
                    (df["Patient_Name"].str.lower() == pname.lower())
                ])


    '''

Data preprocessing complete ✓
Dataset shape: (1000, 11)
Risk distribution:
risk
0    558
1    442
Name: count, dtype: int64
Model trained and saved ✓
Test metrics: {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}
